In [0]:
bronze_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/Volumes/workspace/default/vol1/source_data.csv")
bronze_df.write.format("delta").mode("overwrite").saveAsTable("engine_analysis_bronze")

display(bronze_df.limit(5))

In [0]:
from pyspark.sql.functions import expr

silver_df = bronze_df.drop("op_set_2")
silver_df = silver_df.withColumn("oph", expr("try_cast(oph as int)")).dropna()
silver_df = silver_df.withColumnRenamed("oph", "operating_hours") \
                     .withColumnRenamed("pist_m", "piston_material") \
                     .withColumnRenamed("bmep", "brake_mean_effective_pressure") \
                     .withColumnRenamed("ng_imp", "natural_gas_impurities") \
                     .withColumnRenamed("past_dmg", "has_past_damage") \
                     .withColumnRenamed("number_up", "unplanned_events_count") \
                     .withColumnRenamed("number_tc", "turbo_chargers_count") \
                     .withColumnRenamed("op_set_1", "operational_engine_setting_1") \
                     .withColumnRenamed("op_set_3", "operational_engine_setting_3")
silver_df.write.format("delta").mode("overwrite").saveAsTable("engine_analysis_silver")

display(silver_df.limit(5))

In [0]:
from pyspark.sql.functions import col

gold_df = silver_df.drop("operational_engine_setting_1","operational_engine_setting_3")
gold_df = gold_df.filter(
    (col("operating_hours") <= 120000) &
    (col("operating_hours") >= 0) &
    (col("piston_material") >= 0) &
    (col("brake_mean_effective_pressure") >= 0) & 
    (col("natural_gas_impurities") >= 0) &
    (col("rpm_max") >= 0) &
    (col("unplanned_events_count") >= 0) &
    (col("turbo_chargers_count") >= 0) &
    (col("resting_analysis_results").isin(0, 1, 2)) &
    (col("has_past_damage").isin(0, 1)) &
    (col("full_load_issues").isin(0, 1)) &
    (col("high_breakdown_risk").isin(0, 1)) &
    (col("issue_type").isin(
        "typical", 
        "atypical", 
        "non-related", 
        "non-symptomatic")))
gold_df.write.format("delta").mode("overwrite").saveAsTable("engine_analysis_gold")

display(gold_df.limit(5))